In [1]:
# pip install yfinance pandas matplotlib seaborn pandas-datareader

In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
from pandas_datareader import data as pdr

In [3]:
btc = yf.download("BTC-USD", start="2014-01-01")
gold = yf.download("GC=F", start="2014-01-01")
sp500 = yf.download("^GSPC", start="2014-01-01")
cpi = pdr.DataReader("CPIAUCSL", "fred", start="2014-01-01")
fed_rate = pdr.DataReader("FEDFUNDS", "fred", start="2014-01-01")
vix = yf.download("^VIX", start="2014-01-01")
if isinstance(sp500.columns, pd.MultiIndex):
    sp500.columns = sp500.columns.droplevel(1)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [4]:
sp500.to_csv("SP500_data.csv")

In [5]:
btc = pd.read_csv("Bitcoin_data.csv")
gold = pd.read_csv("Gold_data.csv")
sp500 = pd.read_csv("SP500_data.csv", skiprows=3, names=['Date', 'Close', 'High', 'Low', 'Open', 'Volume'])
fed = pd.read_csv("Federal_Funds_Rate_data.csv")
cpi = pd.read_csv("CPI_data.csv")
vix = pd.read_csv("vix.csv")
# sp500.index = pd.to_datetime(sp500.index, errors="coerce")
# sp500['Date'] = pd.to_datetime(sp500['Date'])
# Set the index to Date
# sp500.set_index('Date', inplace=True)
# Select only the Close column and rename it
# sp500 = sp500[['Close']].rename(columns={'Close': 'SP500'})

In [ ]:
#sp500.columns


# sp500 = sp500.reset_index().rename(columns={"index": "Date"})
# sp500['Date'] = pd.to_datetime(sp500['Date'], errors='coerce', format='%Y-%m-%d')
# sp500 = sp500.dropna(subset=['Date'])

In [6]:
btc['Date'] = pd.to_datetime(btc['Date'])
gold['Date'] = pd.to_datetime(gold['Date'])
sp500['Date'] = pd.to_datetime(sp500['Date'])

fed['Date'] = pd.to_datetime(fed['Date'])
cpi['Date'] = pd.to_datetime(cpi['Date'])


In [7]:
vix = vix.iloc[2:]   # remove first 2 rows
vix.columns = ['Date','Close','High','Low','Open','Volume']

vix['Date'] = pd.to_datetime(vix['Date'])
vix.set_index('Date', inplace=True)

vix = vix[['Close']].rename(columns={'Close':'VIX'})


In [8]:
btc.set_index('Date', inplace=True)
gold.set_index('Date', inplace=True)
sp500.set_index('Date', inplace=True)

fed.set_index('Date', inplace=True)
cpi.set_index('Date', inplace=True)


In [9]:
# btc = btc.loc[start_date:]
# gold = gold.loc[start_date:]
# sp500 = sp500.loc[start_date:]
# vix = vix.loc[start_date:]
fed_daily = fed.resample("D").ffill()
cpi_daily = cpi.resample("D").ffill()



In [10]:
print(btc.columns)
print(gold.columns)
print(sp500.columns)


Index(['Close', 'High', 'Low', 'Open', 'Volume'], dtype='object')
Index(['Close', 'High', 'Low', 'Open', 'Volume'], dtype='object')
Index(['Close', 'High', 'Low', 'Open', 'Volume'], dtype='object')


In [11]:
btc = btc[['Close']].rename(columns={'Close':'BTC'})
gold = gold[['Close']].rename(columns={'Close':'Gold'})
sp500 = sp500[['Close']].rename(columns={'Close':'SP500'})

fed_daily.rename(columns={'Federal_Funds_Rate':'FED_RATE'}, inplace=True)
cpi_daily.rename(columns={'CPI_Index':'CPI'}, inplace=True)


In [12]:
start_date = "2014-01-01"

btc = btc.loc[start_date:]
gold = gold.loc[start_date:]
sp500 = sp500.loc[start_date:]
vix = vix.loc[start_date:]
fed_daily = fed_daily.loc[start_date:]
cpi_daily = cpi_daily.loc[start_date:]


In [13]:
data = btc.join([
    gold,
    sp500,
    vix,
    fed_daily,
    cpi_daily
], how='outer')


In [14]:
data = data.sort_index()


In [15]:
data = data.loc["2014-01-01":]


In [16]:
print(data.shape)


(4451, 6)


In [17]:
data[['FED_RATE','CPI']] = data[['FED_RATE','CPI']].ffill()


In [18]:
data.to_csv("raw_dataset.csv")
